In [16]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [41]:
df=pd.read_csv('garments_worker_productivity.csv')
#df.head()
#df.info()
#df.describe()
df.isnull().sum()
df.nunique()


date                      59
quarter                    5
department                 3
day                        6
team                      12
targeted_productivity      9
smv                       70
wip                      548
over_time                143
incentive                 48
idle_time                 12
idle_men                  10
no_of_style_change         3
no_of_workers             61
actual_productivity      879
dtype: int64

In [42]:
print("WIP Mean:", df["wip"].mean())
print("WIP Median:", df["wip"].median())
print("WIP Minimum:", df["wip"].min())
print("WIP Maximum:", df["wip"].max())

WIP Mean: 1190.4659913169319
WIP Median: 1039.0
WIP Minimum: 7.0
WIP Maximum: 23122.0


In [43]:
df["MeetsTarget"] = (
    df["actual_productivity"] >= df["targeted_productivity"]
).astype(int)
df[[
    "targeted_productivity",
    "actual_productivity",
    "MeetsTarget"
]].head(10)
print(df["MeetsTarget"].value_counts())
print(df["MeetsTarget"].value_counts(normalize=True) * 100)

MeetsTarget
1    875
0    322
Name: count, dtype: int64
MeetsTarget
1    73.099415
0    26.900585
Name: proportion, dtype: float64


In [ ]:
df["date"] = pd.to_datetime(df["date"])
df["month"] = df["date"].dt.month
df["day_of_month"] = df["date"].dt.day
df = df.drop(columns=["date"])


In [48]:
df['department'] = df['department'].str.strip()
df.head()

,quarter,department,day,team,targeted_productivity,smv,wip,over_time,incentive,idle_time,idle_men,no_of_style_change,no_of_workers,actual_productivity,MeetsTarget,month,day_of_month
0,Quarter1,sweing,Thursday,8,0.80,26.16,1108.0,7080,98,0.0,0,0,59.0,0.940725,1,1,1
1,Quarter1,finishing,Thursday,1,0.75,3.94,NaN,960,0,0.0,0,0,8.0,0.886500,1,1,1
2,Quarter1,sweing,Thursday,11,0.80,11.41,968.0,3660,50,0.0,0,0,30.5,0.800570,1,1,1
3,Quarter1,sweing,Thursday,12,0.80,11.41,968.0,3660,50,0.0,0,0,30.5,0.800570,1,1,1
4,Quarter1,sweing,Thursday,6,0.80,25.90,1170.0,1920,50,0.0,0,0,56.0,0.800382,1,1,1


In [46]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["actual_productivity", "MeetsTarget"])

y_reg = df["actual_productivity"]
y_class = df["MeetsTarget"]


categorical_features = [
    "quarter",
    "department",
    "day",
    "team"
]

numerical_features = [
    "targeted_productivity",
    "smv",
    "wip",
    "over_time",
    "incentive",
    "idle_time",
    "idle_men",
    "no_of_style_change",
    "no_of_workers",
    "month",
    "day_of_month"
]


X_train, X_test, y_reg_train, y_reg_test, y_class_train, y_class_test = train_test_split(
    X,
    y_reg,
    y_class,
    test_size=0.2,
    random_state=42,
    stratify=y_class
)


print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_reg_train:", y_reg_train.shape)
print("y_reg_test:", y_reg_test.shape)
print("y_class_train:", y_class_train.shape)
print("y_class_test:", y_class_test.shape)

X_train: (957, 15)
X_test: (240, 15)
y_reg_train: (957,)
y_reg_test: (240,)
y_class_train: (957,)
y_class_test: (240,)


In [49]:
X_train.head()

,quarter,department,day,team,targeted_productivity,smv,wip,over_time,incentive,idle_time,idle_men,no_of_style_change,no_of_workers,month,day_of_month
1109,Quarter2,finishing,Sunday,1,0.75,3.94,NaN,960,0,0.0,0,0,8.0,3,8
1000,Quarter1,finishing,Sunday,2,0.70,3.90,NaN,960,0,0.0,0,0,8.0,3,1
468,Quarter4,finishing,Tuesday,2,0.75,3.94,NaN,1200,0,0.0,0,0,10.0,1,27
1160,Quarter2,sweing,Tuesday,11,0.80,11.41,338.0,3420,50,0.0,0,0,58.0,3,10
93,Quarter1,sweing,Tuesday,4,0.80,28.08,925.0,10350,50,0.0,0,0,57.5,1,6


In [50]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression

# 1. Imputation Transformer (Imputes missing WIP at index 6)
trf1 = ColumnTransformer([
    ('impute_wip', SimpleImputer(strategy='median'), [6])
], remainder='passthrough')

# 2. One-Hot Encoding Transformer (Encodes categoricals shifted to indices [1, 2, 3, 4])
trf2 = ColumnTransformer([
    ('ohe_cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), [1, 2, 3, 4])
], remainder='passthrough')

# 3. Scaling Transformer (Scales all generated numerical features)
trf3 = ColumnTransformer([
    ('scale', StandardScaler(), slice(0, None))
])

# 4. Model Estimators
trf4_lr = LinearRegression()
trf4_clf = LogisticRegression(max_iter=1000, random_state=42)

In [88]:
# Create Sequential Pipelines
pipe_lr = Pipeline([
    ('trf1', trf1),
    ('trf2', trf2),
    ('trf3', trf3),
    ('trf4', trf4_lr)
])

pipe_clf = Pipeline([
    ('trf1', trf1),
    ('trf2', trf2),
    ('trf3', trf3),
    ('trf4', trf4_clf)
])

import time

# 1. LINEAR REGRESSION PIPELINE (Fit, Predict, Time)

start_lr_train = time.perf_counter()
pipe_lr.fit(X_train, y_reg_train)
skl_lr_train_time = time.perf_counter() - start_lr_train

start_lr_pred = time.perf_counter()
y_reg_pred_skl = pipe_lr.predict(X_test)
skl_lr_pred_time = time.perf_counter() - start_lr_pred


# 2. LOGISTIC REGRESSION PIPELINE (Fit, Predict, Time)

start_clf_train = time.perf_counter()
pipe_clf.fit(X_train, y_class_train)
skl_clf_train_time = time.perf_counter() - start_clf_train

start_clf_pred = time.perf_counter()
y_class_pred_skl = pipe_clf.predict(X_test)
skl_clf_pred_time = time.perf_counter() - start_clf_pred

print("Pipeline fitting, predictions, and timing complete!")
print(y_reg_pred_skl[:5])

Pipeline fitting, predictions, and timing complete!
[0.8361802  0.66008084 0.78127979 0.61110874 0.71437012]


In [53]:
from sklearn.metrics import (
    mean_absolute_error, root_mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score
)

print("=== PART A: Scikit-Learn Linear Regression ===")
print(f"MAE:             {mean_absolute_error(y_reg_test, y_reg_pred_skl):.4f}")
print(f"RMSE:            {root_mean_squared_error(y_reg_test, y_reg_pred_skl):.4f}")
print(f"R2 Score:        {r2_score(y_reg_test, y_reg_pred_skl):.4f}")
print(f"Training Time:   {skl_lr_train_time:.6f} s")
print(f"Prediction Time: {skl_lr_pred_time:.6f} s")

print("\n=== PART A: Scikit-Learn Logistic Regression ===")
print(f"Accuracy:        {accuracy_score(y_class_test, y_class_pred_skl):.4f}")
print(f"Precision:       {precision_score(y_class_test, y_class_pred_skl):.4f}")
print(f"Recall:          {recall_score(y_class_test, y_class_pred_skl):.4f}")
print(f"F1-Score:        {f1_score(y_class_test, y_class_pred_skl):.4f}")
print(f"Training Time:   {skl_clf_train_time:.6f} s")
print(f"Prediction Time: {skl_clf_pred_time:.6f} s")

=== PART A: Scikit-Learn Linear Regression ===
MAE:             0.1075
RMSE:            0.1411
R2 Score:        0.3146
Training Time:   0.237824 s
Prediction Time: 0.026574 s

=== PART A: Scikit-Learn Logistic Regression ===
Accuracy:        0.7083
Precision:       0.7807
Recall:          0.8343
F1-Score:        0.8066
Training Time:   0.098201 s
Prediction Time: 0.019549 s


In [ ]:
import time
start_time=time.perf_counter()

X_train_manual = X_train.copy()
X_test_manual=X_test.copy()
wip_median = X_train_manual["wip"].median()
X_train_manual["wip"]=X_train_manual["wip"].fillna(wip_median)
X_test_manual["wip"]=X_test_manual["wip"].fillna(wip_median)

X_train_manual=pd.get_dummies(X_train_manual, columns=categorical_features, drop_first=True)
X_test_manual=pd.get_dummies(X_test_manual, columns=categorical_features, drop_first=True)


X_test_manual = X_test_manual.reindex(
    columns=X_train_manual.columns,
    fill_value=0
)
X_train_manual = X_train_manual.to_numpy(dtype=float)
X_test_manual = X_test_manual.to_numpy(dtype=float)


print("Training shape:", X_train_manual.shape)
print("Testing shape:", X_test_manual.shape)
train_mean= X_train_manual.mean(axis=0)
train_std= X_train_manual.std(axis=0)
X_train_manual = (X_train_manual - train_mean) / train_std
X_test_manual = (X_test_manual - train_mean) / train_std

class LR:
    def __init__(self):
        self.intercepts= None
        self.coef_ = None
        
    def fit(self, X_train_manual, y_reg_train):
        X_train_manual = np.insert(X_train_manual,0,1,axis=1)
        betas= np.linalg.pinv(np.dot(X_train_manual.T,X_train_manual)).dot(X_train_manual.T).dot(y_reg_train)
        self.intercepts= betas[0]
        self.coef_= betas[1:]  
        

    def predict(self, X_test_manual):
        y_pred=np.dot(X_test_manual,self.coef_)+self.intercepts
        return y_pred


lr=LR()

lr.fit(X_train_manual, y_reg_train.to_numpy())
manual_lr_train_time = time.perf_counter() - start_time

start_time=time.perf_counter()
y_reg_pred_manual=lr.predict(X_test_manual)
manual_lr_pred_time = time.perf_counter() - start_time
print("Intercept:", lr.intercepts)
print("Number of coefficients:", len(lr.coef_))
print("First 5 coefficients:", lr.coef_[:5])
print("First 5 predictions:", y_reg_pred_manual[:5])
print("Training time:", manual_lr_train_time)
print("Prediction time:", manual_lr_pred_time)       


Training shape: (957, 33)
Testing shape: (240, 33)
Intercept: 0.7324859741619651
Number of coefficients: 33
First 5 coefficients: [ 0.06363961 -0.08487672  0.00719535 -0.01411301  0.01106657]
First 5 predictions: [0.8361802  0.66008084 0.78127979 0.61110874 0.71437012]
Training time: 0.09183210000628605
Prediction time: 0.00023199996212497354


In [89]:
x=r2_score(y_reg_test, y_reg_pred_manual)
MAE = np.mean(np.abs(y_reg_test.to_numpy() - y_reg_pred_manual ))
RMSE = (np.mean((y_reg_test.to_numpy() - y_reg_pred_manual) ** 2))**(1/2)

print(MAE, RMSE,x)

0.10753278166336365 0.14108307876177523 0.31458343175628245


In [90]:
class LogisticRegressionManual:

    def __init__(self, learning_rate=0.01, n_iterations=5000):
        self.learning_rate = learning_rate
        self.n_iterations = n_iterations
        self.weights = None
        self.bias = 0

    def sigmoid(self, z):
        return 1 / (1 + np.exp(-z))

    def fit(self, X_train, y_train):

        n_samples, n_features = X_train.shape

        # Initialize weights and bias
        self.weights = np.zeros(n_features)
        self.bias = 0

        # Gradient Descent
        for i in range(self.n_iterations):

            # Linear equation
            z = X_train @ self.weights + self.bias

            # Convert to probability
            y_pred = self.sigmoid(z)

            # Calculate gradients
            dw = (1 / n_samples) * (X_train.T @ (y_pred - y_train))
            db = (1 / n_samples) * np.sum(y_pred - y_train)

            # Update weights and bias
            self.weights = self.weights - self.learning_rate * dw
            self.bias = self.bias - self.learning_rate * db

    def predict_proba(self, X_test):

        z = X_test @ self.weights + self.bias

        return self.sigmoid(z)

    def predict(self, X_test):

        probabilities = self.predict_proba(X_test)

        return (probabilities >= 0.5).astype(int)

In [91]:
import time

model_log = LogisticRegressionManual(
    learning_rate=0.01,
    n_iterations=5000
)

start_time = time.perf_counter()

model_log.fit(
    X_train_manual,
    y_class_train.to_numpy()
)

manual_log_train_time = time.perf_counter() - start_time


start_time = time.perf_counter()

y_class_pred_manual = model_log.predict(X_test_manual)

manual_log_pred_time = time.perf_counter() - start_time


print("First 10 predictions:")
print(y_class_pred_manual[:10])

print("\nTraining Time:", manual_log_train_time)
print("Prediction Time:", manual_log_pred_time)

First 10 predictions:
[1 1 1 0 1 1 1 1 0 0]

Training Time: 0.411660399986431
Prediction Time: 0.006666300003416836


In [92]:
y_true = y_class_test.to_numpy()
y_pred = y_class_pred_manual

# Calculate TP, TN, FP, FN
tp = np.sum((y_true == 1) & (y_pred == 1))
tn = np.sum((y_true == 0) & (y_pred == 0))
fp = np.sum((y_true == 0) & (y_pred == 1))
fn = np.sum((y_true == 1) & (y_pred == 0))

# Accuracy
accuracy = (tp + tn) / (tp + tn + fp + fn)

# Precision
precision = tp / (tp + fp) if (tp + fp) > 0 else 0

# Recall
recall = tp / (tp + fn) if (tp + fn) > 0 else 0

# F1-score
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print("TP:", tp)
print("TN:", tn)
print("FP:", fp)
print("FN:", fn)

print("\nManual Logistic Regression Metrics")
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-Score:", f1)

TP: 148
TN: 21
FP: 44
FN: 27

Manual Logistic Regression Metrics
Accuracy: 0.7041666666666667
Precision: 0.7708333333333334
Recall: 0.8457142857142858
F1-Score: 0.8065395095367848


In [93]:
comparison_reg = pd.DataFrame({
    "Metric": [
        "MAE",
        "RMSE",
        "R2 Score",
        "Training Time (s)",
        "Prediction Time (s)"
    ],
    
    "Scikit-Learn": [
        mean_absolute_error(y_reg_test, y_reg_pred_skl),
        root_mean_squared_error(y_reg_test, y_reg_pred_skl),
        r2_score(y_reg_test, y_reg_pred_skl),
        skl_lr_train_time,
        skl_lr_pred_time
    ],
    
    "From Scratch": [
        MAE,
        RMSE,
        x,
        manual_lr_train_time,
        manual_lr_pred_time
    ]
})

comparison_reg

,Metric,Scikit-Learn,From Scratch
0,MAE,0.107533,0.107533
1,RMSE,0.141083,0.141083
2,R2 Score,0.314583,0.314583
3,Training Time (s),0.515098,0.091832
4,Prediction Time (s),0.047399,0.000232


In [94]:
comparison_class = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1-Score",
        "Training Time (s)",
        "Prediction Time (s)"
    ],

    "Scikit-Learn": [
        accuracy_score(y_class_test, y_class_pred_skl),
        precision_score(y_class_test, y_class_pred_skl),
        recall_score(y_class_test, y_class_pred_skl),
        f1_score(y_class_test, y_class_pred_skl),
        skl_clf_train_time,
        skl_clf_pred_time
    ],

    "From Scratch": [
        accuracy,
        precision,
        recall,
        f1,
        manual_log_train_time,
        manual_log_pred_time
    ]
})

comparison_class

,Metric,Scikit-Learn,From Scratch
0,Accuracy,0.708333,0.704167
1,Precision,0.780749,0.770833
2,Recall,0.834286,0.845714
3,F1-Score,0.806630,0.806540
4,Training Time (s),0.148272,0.411660
5,Prediction Time (s),0.014792,0.006666


In [95]:
class LogisticRegressionOptimized:

    def __init__(self, learning_rate=0.01, n_iterations=5000, lambda_=0.01):
        self.learning_rate = learning_rate
        self.n_iterations = n_iterations
        self.lambda_ = lambda_
        self.weights = None
        self.bias = 0

    def sigmoid(self, z):
        z = np.clip(z, -500, 500)
        return 1 / (1 + np.exp(-z))

    def fit(self, X_train, y_train):

        n_samples, n_features = X_train.shape

        self.weights = np.zeros(n_features)
        self.bias = 0

        for i in range(self.n_iterations):

            z = X_train @ self.weights + self.bias
            y_pred = self.sigmoid(z)

            dw = (1 / n_samples) * (
                X_train.T @ (y_pred - y_train)
            )

            # L2 regularization
            dw += (self.lambda_ / n_samples) * self.weights

            db = (1 / n_samples) * np.sum(y_pred - y_train)

            self.weights -= self.learning_rate * dw
            self.bias -= self.learning_rate * db

    def predict_proba(self, X_test):

        z = X_test @ self.weights + self.bias

        return self.sigmoid(z)

    def predict(self, X_test):

        probabilities = self.predict_proba(X_test)

        return (probabilities >= 0.5).astype(int)

In [112]:
optimized_log = LogisticRegressionOptimized(
    learning_rate=0.01,
    n_iterations=15000,
    lambda_=0.01
)

start_time = time.perf_counter()

optimized_log.fit(
    X_train_manual,
    y_class_train.to_numpy()
)

optimized_log_train_time = time.perf_counter() - start_time


start_time = time.perf_counter()

y_class_pred_optimized = optimized_log.predict(X_test_manual)

optimized_log_pred_time = time.perf_counter() - start_time


print("First 10 optimized predictions:")
print(y_class_pred_optimized[:10])

print("\nOptimized Training Time:", optimized_log_train_time)
print("Optimized Prediction Time:", optimized_log_pred_time)

First 10 optimized predictions:
[1 1 1 0 1 1 0 1 0 0]

Optimized Training Time: 1.28740899998229
Optimized Prediction Time: 0.002680600038729608


In [113]:
y_true = y_class_test.to_numpy()
y_pred = y_class_pred_optimized

# Calculate TP, TN, FP, FN
tp_opt = np.sum((y_true == 1) & (y_pred == 1))
tn_opt = np.sum((y_true == 0) & (y_pred == 0))
fp_opt = np.sum((y_true == 0) & (y_pred == 1))
fn_opt = np.sum((y_true == 1) & (y_pred == 0))

# Accuracy
accuracy_opt = (tp_opt + tn_opt) / (tp_opt + tn_opt + fp_opt + fn_opt)

# Precision
precision_opt = tp_opt / (tp_opt + fp_opt) if (tp_opt + fp_opt) > 0 else 0

# Recall
recall_opt = tp_opt / (tp_opt + fn_opt) if (tp_opt + fn_opt) > 0 else 0

# F1-score
f1_opt = (
    2 * precision_opt * recall_opt / (precision_opt + recall_opt)
    if (precision_opt + recall_opt) > 0 else 0
)

print("Optimized Logistic Regression Metrics")
print("Accuracy:", accuracy_opt)
print("Precision:", precision_opt)
print("Recall:", recall_opt)
print("F1-Score:", f1_opt)

Optimized Logistic Regression Metrics
Accuracy: 0.7041666666666667
Precision: 0.776595744680851
Recall: 0.8342857142857143
F1-Score: 0.8044077134986227


In [114]:
classification_final = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1-Score",
        "Training Time (s)",
        "Prediction Time (s)"
    ],

    "Scikit-Learn": [
        accuracy_score(y_class_test, y_class_pred_skl),
        precision_score(y_class_test, y_class_pred_skl),
        recall_score(y_class_test, y_class_pred_skl),
        f1_score(y_class_test, y_class_pred_skl),
        skl_clf_train_time,
        skl_clf_pred_time
    ],

    "From Scratch": [
        accuracy,
        precision,
        recall,
        f1,
        manual_log_train_time,
        manual_log_pred_time
    ],

    "Optimized From Scratch": [
        accuracy_opt,
        precision_opt,
        recall_opt,
        f1_opt,
        optimized_log_train_time,
        optimized_log_pred_time
    ]
})

classification_final

,Metric,Scikit-Learn,From Scratch,Optimized From Scratch
0,Accuracy,0.708333,0.704167,0.704167
1,Precision,0.780749,0.770833,0.776596
2,Recall,0.834286,0.845714,0.834286
3,F1-Score,0.806630,0.806540,0.804408
4,Training Time (s),0.148272,0.411660,1.287409
5,Prediction Time (s),0.014792,0.006666,0.002681


In [99]:
regression_final = pd.DataFrame({
    "Metric": [
        "MAE",
        "RMSE",
        "R2 Score",
        "Training Time (s)",
        "Prediction Time (s)"
    ],

    "Scikit-Learn": [
        mean_absolute_error(y_reg_test, y_reg_pred_skl),
        root_mean_squared_error(y_reg_test, y_reg_pred_skl),
        r2_score(y_reg_test, y_reg_pred_skl),
        skl_lr_train_time,
        skl_lr_pred_time
    ],

    "From Scratch": [
        MAE,
        RMSE,
        x,
        manual_lr_train_time,
        manual_lr_pred_time
    ]
})

regression_final

,Metric,Scikit-Learn,From Scratch
0,MAE,0.107533,0.107533
1,RMSE,0.141083,0.141083
2,R2 Score,0.314583,0.314583
3,Training Time (s),0.515098,0.091832
4,Prediction Time (s),0.047399,0.000232
